# 03 — Gold publication: master-data reference tier

| Field | Value |
| ----- | ----- |
| **Sprint** | Sprint 09 — T2.5 |
| **Layer** | `gold/reference/` |
| **Source** | `Tables/silver/master-data/<table>/` (Delta, validated) |
| **Target** | `Tables/gold/reference/<table>/` (Delta, published) |
| **Governance** | [ADR-0015](../../../docs/adr/0015-skip-sql-for-mvp-demo.md), [ADR-0016](../../../docs/adr/0016-no-phi-in-mvp-demo-scope.md), [ADR-0013](../../../docs/adr/0013-temporary-us-region-demo-scope.md) |
| **Design spec** | [sprint-09 §1.2 governance columns + §2.2 Notebook 03](../../../docs/sprints/sprint-09-master-data-simulation-and-capacity-dashboard.md) |

## Purpose

Publish the 9 validated silver dims/facts into the `gold/reference/` tier with
the full governance column contract from sprint-09 §1.2. Overwrite-mode Delta
write keeps the notebook idempotent — reruns produce the same gold state.

## Mandatory governance columns (sprint-09 §1.2)

| Column | Value strategy |
| ------ | -------------- |
| `_classification` | Constant `Operational confidential` |
| `_residency_tag` | Preserved from silver (Gate 4 already asserted `{CH-North, US-West}`) |
| `_legal_basis` | Constant `nDSG/KVG` |
| `_retention_class` | Constant `R3` (7 years operational) |
| `_data_quality` | Preserved from silver (Gate 5 already asserted `{explicit, inferred, missing}`) |
| `_lineage_ref` | Rewritten to `silver:<table>:<gold_ts>` (silver→gold hop) |
| `_pseudonymisation_flag` | Constant `false` — no PII in reference data (ADR-0016 §Decision 1) |

In [ ]:
target_lakehouse = 'lh_ihzhhpf_sit'
silver_root = 'Tables/silver/master-data'
gold_root = 'Tables/gold/reference'
run_id = 'run-manual-local'
log_analytics_workspace_id = None  # optional; if None, summary is only printed

In [ ]:
# Master-data table registry (matches bronze + silver notebooks).
TABLES = [
    'dim_hospital',
    'dim_specialty',
    'dim_hospital_service',
    'dim_disease',
    'dim_treatment',
    'dim_drg',
    'dim_ward_capacityunit',
    'fact_capacity_baseline',
    'map_disease_treatment_specialty_service',
]

In [ ]:
from datetime import datetime, timezone
from pyspark.sql import DataFrame, functions as F

GOVERNANCE_CONSTANTS = {
    '_classification':        'Operational confidential',
    '_legal_basis':           'nDSG/KVG',
    '_retention_class':       'R3',
    '_pseudonymisation_flag': False,
}

def _now_iso() -> str:
    return datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

def stamp_governance(df: DataFrame, table: str, gold_ts: str) -> DataFrame:
    """Apply the 7-column governance contract from sprint-09 §1.2 to a silver DataFrame."""
    out = df
    # Constants (idempotent overwrite — silver may or may not already carry these).
    for col, val in GOVERNANCE_CONSTANTS.items():
        out = out.withColumn(col, F.lit(val))
    # Preserve silver-derived tags if present; otherwise default to demo-scope safe values.
    if '_residency_tag' not in out.columns:
        out = out.withColumn('_residency_tag', F.lit('US-West'))
    if '_data_quality' not in out.columns:
        out = out.withColumn('_data_quality', F.lit('explicit'))
    # Rewrite lineage to record the silver→gold hop.
    out = out.withColumn('_lineage_ref', F.lit(f'silver:{table}:{gold_ts}'))
    return out

In [ ]:
def publish_table(table: str, gold_ts: str):
    src = f'{silver_root}/{table}'
    tgt = f'{gold_root}/{table}'
    df = spark.read.format('delta').load(src)
    out = stamp_governance(df, table, gold_ts)
    (out.write
        .format('delta')
        .mode('overwrite')
        .option('overwriteSchema', 'true')
        .save(tgt))
    total = out.count()
    quality_dist = {r['_data_quality']: r['count'] for r in
                    out.groupBy('_data_quality').count().collect()}
    residency_dist = {r['_residency_tag']: r['count'] for r in
                      out.groupBy('_residency_tag').count().collect()}
    return total, quality_dist, residency_dist

In [ ]:
gold_ts = _now_iso()
results = []
for table in TABLES:
    total, quality_dist, residency_dist = publish_table(table, gold_ts)
    results.append({
        'table': table,
        'row_count': total,
        'quality_distribution': quality_dist,
        'residency_distribution': residency_dist,
    })

print('Gold publication summary (run_id=%s, gold_ts=%s)' % (run_id, gold_ts))
print('-' * 88)
for r in results:
    print(f"{r['table']:<48s} rows={r['row_count']:<6d} quality={r['quality_distribution']} residency={r['residency_distribution']}")

In [ ]:
# Log Analytics emit — stub for local runs; wired to the workspace DCE/DCR when
# `log_analytics_workspace_id` is provided by the calling Fabric pipeline.
import json as _json

payload = {
    'run_id': run_id,
    'gold_ts': gold_ts,
    'stage': 'gold_reference_publication',
    'tables': results,
}

if log_analytics_workspace_id:
    # In Fabric: use notebookutils.credentials or a DCR-bound HTTPS ingestion endpoint.
    # Kept as a printed payload here so this notebook is safe to run without a workspace binding.
    print(f'LOG_ANALYTICS_EMIT workspace={log_analytics_workspace_id}: {_json.dumps(payload)}')
else:
    print('LOG_ANALYTICS_EMIT (dry-run — no workspace bound):')
    print(_json.dumps(payload, indent=2))